# ERA5 Hourly Downloader for Bexar County (2019–2024)

This notebook downloads **ERA5 reanalysis hourly single-level data** for the Bexar County bounding box:

- **North:** 29.60  
- **South:** 29.30  
- **West:** -98.65  
- **East:** -98.30  

Variables included:

- 10m u-component of wind
- 10m v-component of wind
- 2m dewpoint temperature
- 2m temperature
- Surface pressure
- Total precipitation
- Boundary layer height

It downloads data in **monthly chunks** for **2019–2024**.


## Before you run

Make sure you have:

1. Installed the CDS API package:
```python
!pip install cdsapi
```

2. Created your `.cdsapirc` file in your home directory.

On Windows, usually:
```text
C:\Users\YOUR_USERNAME\.cdsapirc
```

Example content:
```text
url: https://cds.climate.copernicus.eu/api
key: YOUR_API_KEY
```

3. Accepted the ERA5 dataset terms on the CDS website.


In [ ]:
import calendar
import concurrent.futures as cf
import time
from pathlib import Path

import cdsapi


In [ ]:
# =========================
# USER SETTINGS
# =========================
OUTPUT_DIR = Path('era5_bexar_hourly_2019_2024')
START_YEAR = 2019
END_YEAR = 2024

# CDS area format: [North, West, South, East]
AREA = [29.60, -98.65, 29.30, -98.30]

VARIABLES = [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '2m_dewpoint_temperature',
    '2m_temperature',
    'surface_pressure',
    'total_precipitation',
    'boundary_layer_height',
]

# Use 'netcdf' for easier xarray usage.
# Change to 'grib' if you want smaller/raw files.
DATA_FORMAT = 'netcdf'

# Keep parallelism modest to avoid CDS queue issues
MAX_WORKERS = 3

# Retry settings
MAX_RETRIES = 5
RETRY_SLEEP_SECONDS = 20

DATASET = 'reanalysis-era5-single-levels'


In [ ]:
def all_hours():
    return [f'{h:02d}:00' for h in range(24)]


def days_in_month(year, month):
    n_days = calendar.monthrange(year, month)[1]
    return [f'{d:02d}' for d in range(1, n_days + 1)]


def target_path(year, month):
    ext = 'nc' if DATA_FORMAT == 'netcdf' else 'grib'
    return OUTPUT_DIR / f'era5_bexar_{year}_{month:02d}.{ext}'


def build_request(year, month):
    return {
        'product_type': ['reanalysis'],
        'variable': VARIABLES,
        'year': [str(year)],
        'month': [f'{month:02d}'],
        'day': days_in_month(year, month),
        'time': all_hours(),
        'data_format': DATA_FORMAT,
        'download_format': 'unarchived',
        'area': AREA,
    }


def download_month(year, month):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_file = target_path(year, month)

    if out_file.exists() and out_file.stat().st_size > 0:
        return year, month, f'SKIP (already exists): {out_file.name}'

    request = build_request(year, month)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f'[START] {year}-{month:02d} | attempt {attempt}')
            client = cdsapi.Client(timeout=600, quiet=False, debug=False)
            client.retrieve(DATASET, request, str(out_file))

            if out_file.exists() and out_file.stat().st_size > 0:
                size_mb = out_file.stat().st_size / (1024 * 1024)
                return year, month, f'DONE: {out_file.name} ({size_mb:.1f} MB)'

            raise RuntimeError('Download finished but file missing/empty.')

        except Exception as e:
            print(f'[ERROR] {year}-{month:02d} attempt {attempt}: {e}')
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP_SECONDS * attempt)
            else:
                return year, month, f'FAILED after {MAX_RETRIES} attempts: {e}'

    return year, month, 'FAILED (unexpected fallthrough)'


def build_jobs(start_year, end_year):
    jobs = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            jobs.append((year, month))
    return jobs


In [ ]:
jobs = build_jobs(START_YEAR, END_YEAR)
print(f'Total monthly jobs: {len(jobs)}')
print(jobs[:6], '...')


In [ ]:
# Run this cell to start downloading
results = []

with cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {
        executor.submit(download_month, year, month): (year, month)
        for year, month in jobs
    }

    for future in cf.as_completed(future_map):
        year, month = future_map[future]
        try:
            result = future.result()
            results.append(result)
            print(f'[RESULT] {result[0]}-{result[1]:02d} | {result[2]}')
        except Exception as e:
            print(f'[CRASH] {year}-{month:02d}: {e}')


In [ ]:
# Summary
for year, month, status in sorted(results):
    print(f'{year}-{month:02d} | {status}')


## Optional: Open a downloaded NetCDF file

Use this only after some monthly files have finished downloading.


In [ ]:
import xarray as xr

# Example file
sample_file = OUTPUT_DIR / 'era5_bexar_2019_01.nc'

if sample_file.exists():
    ds = xr.open_dataset(sample_file)
    print(ds)
    print('\nVariables:')
    print(list(ds.data_vars))
else:
    print(f'File not found yet: {sample_file}')
